In [1]:
import json
import requests
import pandas as pd
from datetime import datetime, timedelta

/Users/anna-moe/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
from config_compliance_api import COMPLIANCE_API_TOKEN, WORKSPACE_ID, SINCE_TIMESTAMP

## STEP 3: Extract responses from ChatGPT Compliance API

In [ ]:
# API configuration
headers = {
    "Authorization": f"Bearer {COMPLIANCE_API_TOKEN}",
    "Content-Type": "application/json"
}

# Build API URL
api_url = f"https://api.chatgpt.com/v1/compliance/workspaces/{WORKSPACE_ID}/conversations"
params = {
    "since_timestamp": SINCE_TIMESTAMP,
    # "users": USER_ID
}

print(f"Fetching conversations from: {api_url}")
print(f"Since timestamp: {SINCE_TIMESTAMP} ({datetime.fromtimestamp(SINCE_TIMESTAMP)})")

# Make API request
response = requests.get(api_url, headers=headers, params=params)
print(f"\nAPI Response Status: {response.status_code}")

if response.status_code == 200:
    conversations_json = response.json()
    print(f"✓ Successfully fetched conversations")
    print(f"  Response keys: {list(conversations_json.keys())}")
    
    # Save raw response for inspection
    with open("conversations_raw.json", "w") as f:
        json.dump(conversations_json, f, indent=2)
    print(f"✓ Saved raw response to: conversations_raw.json")
    
else:
    print(f"✗ API request failed: {response.status_code}")
    print(f"Response: {response.text}")
    conversations_json = None

Fetching conversations from: https://api.chatgpt.com/v1/compliance/workspaces/6ef27957-5bbb-4380-be11-4220c30c4176/conversations
Since timestamp: 1770940800 (2026-02-13 08:00:00)

API Response Status: 200
✓ Successfully fetched conversations
  Response keys: ['object', 'data', 'cursor', 'last_id', 'has_more']
✓ Saved raw response to: conversations_raw.json


## STEP 4: Parse JSON response into structured format

In [ ]:
# Extract conversations list from the data array
conv_list = conversations_json.get("data", [])
print(f"Found {len(conv_list)} conversations")

# Parse conversations into flat structure
parsed_messages = []

for conv in conv_list:
    conv_id = conv.get("id", "unknown")
    conv_title = conv.get("title", "")
    conv_created = conv.get("created_at")
    
    # Extract messages from the nested data structure
    messages_container = conv.get("messages", {})
    messages = messages_container.get("data", []) if isinstance(messages_container, dict) else []
    
    # Parse each message
    for idx, msg in enumerate(messages):
        # Get the author role
        author = msg.get("author", {})
        role = author.get("role", "unknown") if isinstance(author, dict) else "unknown"
        
        # Only process user and assistant messages
        if role not in ["user", "assistant"]:
            continue
        
        # Extract content from the value field
        content_obj = msg.get("content", {})
        text = ""
        if isinstance(content_obj, dict):
            text = content_obj.get("value", "")
        
        # Only add messages with content
        if text and text.strip():
            parsed_messages.append({
                "conversation_id": conv_id,
                "conversation_title": conv_title,
                "message_index": idx,
                "role": role,
                "message_content": text.strip(),
                "gpt_id": msg.get("gpt_id"),
                "timestamp": msg.get("created_at")
            })

# Create DataFrame
responses_df = pd.DataFrame(parsed_messages)
print(f"\n✓ Parsed {len(responses_df)} messages")
print(f"\nRoles found: {responses_df['role'].value_counts().to_dict()}")
print(f"\nDataFrame shape: {responses_df.shape}")
print(f"Columns: {responses_df.columns.tolist()}")

# Display sample
print("\nFirst few messages:")
responses_df.head(10)

Found 147 conversations

✓ Parsed 515 messages

Roles found: {'assistant': 262, 'user': 253}

DataFrame shape: (515, 7)
Columns: ['conversation_id', 'conversation_title', 'message_index', 'role', 'message_content', 'gpt_id', 'timestamp']

First few messages:


,conversation_id,conversation_title,message_index,role,message_content,gpt_id,timestamp
0,698e77fd-7d84-8348-b281-c778312fb50a,Keyboard mishap,3,user,asssssccccccccc,None,1.770945e+09
1,698e77fd-7d84-8348-b281-c778312fb50a,Keyboard mishap,6,assistant,Looks like your keyboard went for a walk 😄 \n...,None,1.770945e+09
2,698e77fd-7d84-8348-b281-c778312fb50a,Keyboard mishap,7,user,what is economics,None,1.770945e+09
3,698e77fd-7d84-8348-b281-c778312fb50a,Keyboard mishap,9,assistant,**Economics** is the study of how people make ...,None,1.770945e+09
4,698e7824-545c-834b-850b-8ce79e6e6a1a,What is Economics,3,user,what is economics,None,1.770945e+09
5,698e7824-545c-834b-850b-8ce79e6e6a1a,What is Economics,6,assistant,"**Economics** is the study of how people, busi...",None,1.770945e+09
6,698e7909-c994-834a-a94e-c78d076770ec,New chat,3,user,Hi,None,1.770945e+09
7,698e7909-c994-834a-a94e-c78d076770ec,New chat,8,assistant,Hello! 😊 \n\nI’m your **H2 Economics learning...,g-6985b604858c81cca91108933f9f9a6f,1.770945e+09
8,698e7979-5404-834b-a644-ce1b39b2e7c6,Economics Concepts and Theories,3,user,"Explain Economics concepts, theories and princ...",None,1.770945e+09
9,698e7979-5404-834b-a644-ce1b39b2e7c6,Economics Concepts and Theories,8,assistant,Great 😊 Let’s start by activating your prior k...,g-6985b604858c81cca91108933f9f9a6f,1.770945e+09


In [ ]:
# Create prompt-response pairs (user message → assistant response)
pairs = []

for conv_id, group in responses_df.groupby("conversation_id"):
    group = group.sort_values("message_index")
    
    current_prompt = None
    for _, row in group.iterrows():
        if row["role"] == "user":
            current_prompt = row
        elif row["role"] == "assistant" and current_prompt is not None:
            pairs.append({
                "conversation_id": conv_id,
                "conversation_title": row["conversation_title"],
                "prompt": current_prompt["message_content"],
                "response": row["message_content"],
                "gpt_id": row["gpt_id"],
                "timestamp": row["timestamp"]
            })
            current_prompt = None

pairs_df = pd.DataFrame(pairs)

print(f"✓ Created {len(pairs_df)} prompt-response pairs")

# Save to CSV
pairs_df.to_csv("conversation_history.csv", index=False, encoding='utf-8-sig')
print(f"✓ Saved to: conversation_history.csv")

# Display sample
print("\nSample pairs:")
pairs_df.head()